## Full Local RAG Pipeline Using FAISS + FLAN-T5-small

In this notebook, we implement a complete local Retrieval-Augmented Generation (RAG) system. This pipeline takes a natural language question, retrieves relevant text chunks using semantic similarity (via FAISS), and generates an answer using a transformer-based language model (FLAN-T5). This is useful for building local document-aware assistants or Q&A systems.

**Step 1: Import Required Libraries**

In [ ]:
# pip install transformers sentencepiece

# Step 1: Import necessary libraries
# Used for similarity search over embeddings.
import faiss
# To handle the chunked text data stored in CSV format.
import pandas as pd
# Converts text into embeddings (vectors).
from sentence_transformers import SentenceTransformer
# Required for numerical operations and array conversion.
import numpy as np
# Loads and runs FLAN-T5 to generate responses.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# Backend engine required for the Hugging Face model inference.
import torch


c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Step 2: Load FAISS Index and Chunked CSV File**

+ I load a pre-built FAISS index that contains embeddings of my PaddleOCR text chunks.
+ The CSV contains the original chunked text data which allows us to map back from the vector to readable context.

In [2]:
# Load FAISS index and chunked text CSV 
faiss_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\faiss_index_PaddleOCR.index"
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\chunked_text_PaddleOCR.csv"

# Load index and chunked data
index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)

**Step 3: Load the Sentence Embedding Model**

+ The same model that was used to generate chunk embeddings must be used to embed the query for accurate semantic comparison.
+ "all-MiniLM-L6-v2" is a lightweight transformer that generates high-quality sentence embeddings.

In [3]:
# Load SentenceTransformer model for query embedding 
model = SentenceTransformer("all-MiniLM-L6-v2")

**Step 4: Define Function to Retrieve Top-k Relevant Chunks**

+ The function turns a user question into a vector and uses FAISS to find the k most similar chunks.
+ The output is a dataframe containing the relevant chunks to be used in the answer generation step.

In [4]:
# Define function to retrieve top-k chunks 
def search_top_k(query, k=2):
    # Convert query into embedding
    query_embedding = model.encode([query])
    
    # Search the index
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k)
    
    # Get the matching rows from the original dataframe
    results = df_chunks.iloc[indices[0]]
    return results[["chunk_id", "filename", "chunk_text"]]

**Step 5: Load the FLAN-T5 Model and Tokenizer**

+ Loads FLAN-T5-small: a lightweight generative model that can run locally.
+ You load both the tokenizer (to encode text into tokens) and the model itself to produce answers.

In [ ]:
# Define the name of the pre-trained model to use
model_name = "google/flan-t5-small"
# Load the tokenizer: this will convert text into tokens (input IDs) the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load the FLAN-T5 model for text generation (sequence-to-sequence)
generation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


**Step 6: Create Prompt from Retrieved Context**

+ Builds an instruction-style prompt that includes both context and the user’s question.
+ The format is designed to guide the model to produce useful, grounded answers.

In [ ]:
# Create prompt using retrieved chunks 
def build_prompt(chunks, question):
    # Join the retrieved text chunks
    context = "\n".join(chunks["chunk_text"].tolist())
    prompt = (
    "You are a helpful assistant. Use the information in the context below to answer the user's question.\n\n"
    "### Context ###\n"
    f"{context}\n\n"
    "### Question ###\n"
    f"{question}\n\n"
    "### Answer ###"
    )
    return prompt

**Step 7: Generate Answer Using FLAN-T5**
+ Converts the prompt into input tokens and feeds it to the FLAN-T5 model.
+ The output is decoded back into readable text.
+ max_tokens=150 controls the length of the generated answer.

In [7]:
# Generate answer using FLAN-T5
def generate_answer(chunks, question, max_tokens=150):
    prompt = build_prompt(chunks, question)

    # Tokenize and encode input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    # Generate response
    #outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    outputs = generation_model.generate(**inputs, max_new_tokens=max_tokens)

    # Decode the output
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


**Step 8: Run a Sample Question**
+ A natural language question is asked.
+ Top relevant chunks are retrieved.
+ An answer is generated based on those chunks.

In [12]:
question = "What did Billy Joel's parents do in the 1930s?"
print(question)
top_chunks = search_top_k(question)
response = generate_answer(top_chunks, question)

What did Billy Joel's parents do in the 1930s?


**Step 9: Print the Answer and Context**
+ Displays the final output from the model.
+ Also shows which chunks were used to answer the question, helpful for debugging or evaluation.

In [10]:
# Print the results
print("💬 Answer:", response)


💬 Answer: [14  a b c d e  Billy Joel '' . Here 's The Thing . WNYC . July 30 , 2012 . Archived from the original on September 22, 2020 . Retrieved October 3,2020 . 16  a b c d e  Billy Joel '' . Here 's The Thing . WNYC . July 30 , 2012 . Archived from the original on September 22 , 2020 . Retrieved October 3,2020 . 16  a b Tallmer , Jerry ( July 22 , 2003 ) . Billy Joel 


In [13]:
print("📄 Context used:")
print(top_chunks["chunk_text"].tolist())

📄 Context used:
["[ 14 . ^ a b c d e `` Billy Joel '' . Here 's The Thing . WNYC . July 30 , 2012 . Archived from the original transcript ) on August 4 , 2012.Retrieved August 3,2012 . `` I grew up on the Island , in the Levittown section of Hicksville.We had a Levitt house , Cape Cod , on the quarter acre . ... My father was a classically trained pianist . He grew up in Nuremberg , Germany , and he also went to school in Switzerland . His father was quite well off . They had a mail- order textile business , Joel Macht Fabrik ... '' 15 . ^ `` Billy Joel 's mom dies at 92 '' . Newsday . Archivedfrom the original on September 22 , 2020 . Retrieved October 3,2020 . 16.^Bego , Mark 2007 ) .Billy Joel : The Biography . Thunder 's Mouth Press . p.13 . ISBN 9781560259893 Charlie Rose.PBS.Archivedfrom the original on March 13 , 2020.Retrieved January 9,2020-via YouTube 18 . ^ a b Tallmer , Jerry ( July 22 , 2003 ) . `` Billy Joel grapples with the past '' . The Villager . Vol . 73 , no . 11 . 